> **Runtime Recommendation:** Use **T4 GPU** runtime in Google Colab for best results.
> TPU runtime is not recommended because TensorFlow's TPU support requires specific
> code modifications (e.g., `TPUStrategy`) and may not be available in all Colab
> environments. The T4 GPU provides 15 GB of VRAM and works seamlessly with standard
> TensorFlow/Keras code.
>
> To select: **Runtime → Change runtime type → T4 GPU**

# Network Intrusion Detection System — CSE-CIC-IDS2018 (Improved Transformer)

This notebook is an **improved version** of `NIDS_Transformer_CICIDS2018.ipynb`.
Key enhancements over the original:

| Area | Improvement |
|---|---|
| Architecture | Features-as-tokens reshape `(N, F, 1)→(N, F, embed_dim)`; `num_layers` blocks actually stacked |
| Training | EarlyStopping, ModelCheckpoint, ReduceLROnPlateau; `class_weight` for imbalance |
| Evaluation | Macro-F1 and Balanced Accuracy added alongside confusion matrix / classification report |
| Reproducibility | Fixed random seeds; TF version + GPU info printed at startup |
| Splits | Stratified train/val/test splits |
| Artifacts | Model, scaler, label classes, and training history saved to `artifacts/` |

## 0. Setup & Reproducibility

In [ ]:
import os
import json
import glob
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score,
    balanced_accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight
import joblib

import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)

%matplotlib inline

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Runtime / GPU check ───────────────────────────────────────────────────────
print(f"TensorFlow version : {tf.__version__}")
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU(s) available   : {[g.name for g in gpus]}")
else:
    print("⚠️  No GPU detected — training will be slower on CPU.")
    print("   Consider enabling a T4 GPU runtime in Colab.")

## 1. Configuration

**Edit only this cell** to adapt the notebook to your environment.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Update to the folder containing CSE-CIC-IDS2018 CSV files.
# Dataset sources:
#   https://www.unb.ca/cic/datasets/ids-2018.html
#   https://www.kaggle.com/datasets/solarmainframe/ids-intrusion-csv
data_dir = "/content/drive/MyDrive/CSE-CIC-IDS2018/"

# Output directory for saved artifacts
ARTIFACT_DIR = "artifacts/cicids2018_transformer"

# ── Data loading ──────────────────────────────────────────────────────────────
# Rows processed per CSV chunk; 50 k keeps peak RAM low on the Colab free tier
CHUNK_SIZE = 50_000
# Maximum rows to load (None = load everything).
# The full dataset has ~16 M rows; 2 M is a practical ceiling for ~12 GB RAM.
MAX_ROWS = 2_000_000

# ── Label column ──────────────────────────────────────────────────────────────
# The target column name in CSE-CIC-IDS2018 CSV files.
LABEL_COL = "Label"

# ── Model hyper-parameters ────────────────────────────────────────────────────
EMBED_DIM    = 32   # projection dim per token (feature)
NUM_HEADS    = 4    # attention heads (must divide EMBED_DIM evenly)
FF_DIM       = 128  # feed-forward inner dimension
NUM_LAYERS   = 2    # number of stacked TransformerBlock layers
DROPOUT_RATE = 0.1

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE          = 256
SHUFFLE_BUFFER_SIZE = 10_000
MAX_EPOCHS          = 30
LEARNING_RATE       = 1e-3
PATIENCE_ES         = 5    # EarlyStopping patience
PATIENCE_LR         = 3    # ReduceLROnPlateau patience

## 2. Mount Google Drive (Colab only)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Load CSE-CIC-IDS2018 Dataset

Data is loaded in chunks to keep peak memory low.  
Cleaning (inf/NaN removal, dtype optimisation) is performed per chunk before accumulation.

> **Deduplication note:** `drop_duplicates()` is applied *per chunk*, which efficiently
> removes exact duplicates that fall within the same chunk window.  
> Cross-chunk duplicates (identical rows split across two consecutive chunks) may still
> remain in the final DataFrame.  To remove them entirely, call
> `data.drop_duplicates(inplace=True)` after the concat — at the cost of higher memory.

In [ ]:
csv_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
if not csv_files:
    raise FileNotFoundError(
        f"No CSV files found in '{data_dir}'. "
        "Please update the 'data_dir' variable in the Configuration cell."
    )
print(f"Found {len(csv_files)} CSV file(s):")
for f in csv_files:
    print(f"  {os.path.basename(f)}")


def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to reduce memory usage."""
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    return df


cleaned_chunks = []
total_rows = 0
stop_loading = False

for f in csv_files:
    if stop_loading:
        break
    for chunk in pd.read_csv(f, low_memory=False, chunksize=CHUNK_SIZE):
        chunk.columns = chunk.columns.str.strip()
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.dropna(inplace=True)
        # Per-chunk dedup (see note above about cross-chunk duplicates)
        chunk.drop_duplicates(inplace=True)
        chunk = optimize_dtypes(chunk)
        cleaned_chunks.append(chunk)
        total_rows += len(chunk)
        if MAX_ROWS is not None and total_rows >= MAX_ROWS:
            stop_loading = True
            break
    print(f"  Loaded {os.path.basename(f)}  (running total: {total_rows:,} rows)")
    if stop_loading:
        print(f"  ↳ Reached MAX_ROWS limit ({MAX_ROWS:,}). Stopping early.")

data = pd.concat(cleaned_chunks, ignore_index=True)
del cleaned_chunks
gc.collect()

# Enforce the hard row cap after concatenation (handles remainder from last chunk)
if MAX_ROWS is not None and len(data) > MAX_ROWS:
    data = data.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)

print(f"\nFinal dataset shape : {data.shape}")
print(f"Memory usage        : {data.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 4. Data Validation & Exploration

In [ ]:
print("Columns:", data.columns.tolist())

# Validate label column
if LABEL_COL not in data.columns:
    raise KeyError(
        f"Label column '{LABEL_COL}' not found in the data. "
        f"Available columns: {data.columns.tolist()}"
    )

print(f"\nUnique labels ({data[LABEL_COL].nunique()}):")
print(data[LABEL_COL].value_counts())

In [ ]:
data.head()

In [ ]:
data.describe()

In [ ]:
label_counts = data[LABEL_COL].value_counts()
plt.figure(figsize=(12, 6))
label_counts.plot(kind="bar")
plt.title("Class Distribution — CSE-CIC-IDS2018")
plt.xlabel("Attack Type")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Feature & Label Preparation

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data[LABEL_COL]).astype("int32")
class_names = label_encoder.classes_.tolist()
num_classes = len(class_names)
print(f"Number of classes : {num_classes}")
print(f"Classes           : {class_names}")

# Drop label column
data.drop(columns=[LABEL_COL], inplace=True)

# Drop non-numeric columns (e.g. Timestamp, flow IDs) — print them explicitly
non_numeric_cols = data.select_dtypes(include=["object", "datetime"]).columns.tolist()
if non_numeric_cols:
    print(f"\nDropping non-numeric columns (cannot be used as model features):")
    for col in non_numeric_cols:
        print(f"  '{col}'  (dtype: {data[col].dtype})")
    print(
        "  Tip: if 'Timestamp' is meaningful, consider extracting numeric features "
        "(hour-of-day, day-of-week) before dropping."
    )
    data.drop(columns=non_numeric_cols, inplace=True)
else:
    print("No non-numeric columns to drop — all remaining columns are numeric.")

print(f"\nFeature matrix shape: {data.shape}")

## 6. Stratified Train / Validation / Test Split

In [ ]:
X = data.values.astype("float32")
del data
gc.collect()

# 80 % train, 10 % val, 10 % test — stratified to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
del X, y
gc.collect()

X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=SEED, stratify=y_test
)

print(f"X_train : {X_train.shape}   y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}     y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}    y_test  : {y_test.shape}")

## 7. Feature Scaling & Reshape for Transformer

The transformer expects a **3-D** input `(batch, seq_len, features)`.  
For tabular data we treat each **feature as a token**, so we reshape
`(N, F)` → `(N, F, 1)`.  
The first `Dense` layer inside the model then projects each token from dim 1 to `EMBED_DIM`.

In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_val   = scaler.transform(X_val).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

# Reshape: (N, num_features) → (N, num_features, 1)  [features as tokens]
X_train = X_train[..., np.newaxis]  # (N, F, 1)
X_val   = X_val[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

num_features = X_train.shape[1]
print(f"X_train shape after reshape : {X_train.shape}")
print(f"num_features (tokens)       : {num_features}")
gc.collect()

## 8. Class Weights (imbalance handling)

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))
print("Class weights (first 10 shown):")
for cls, w in list(class_weight_dict.items())[:10]:
    print(f"  class {cls:2d}  ({class_names[cls]:30s})  weight = {w:.4f}")

## 9. Tabular Transformer Architecture

### Design: features-as-tokens

```
Input  (batch, F, 1)
  │
  ├─ TokenProjection (Dense, dim 1 → EMBED_DIM)   ← one projection per feature
  │     (batch, F, EMBED_DIM)
  │
  ├─ TransformerBlock × NUM_LAYERS
  │     MultiHeadAttention → Add & Norm
  │     Feed-Forward        → Add & Norm
  │
  ├─ GlobalAveragePooling1D  (batch, EMBED_DIM)
  │
  └─ Dense(num_classes, softmax)
```

Attention is computed across the **feature dimension**, so the model learns
which combinations of network-flow features are most informative for each attack type.

In [ ]:
class TransformerBlock(layers.Layer):
    """Single transformer encoder block (Pre-LN variant for training stability)."""

    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout_rate: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout_rate
        )
        self.ffn = tf.keras.Sequential(
            [
                layers.Dense(ff_dim, activation="gelu"),
                layers.Dropout(dropout_rate),
                layers.Dense(embed_dim),
            ]
        )
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        # Pre-LN: normalise before the sublayer
        attn_out = self.attn(self.norm1(x), self.norm1(x), training=training)
        x = x + self.drop1(attn_out, training=training)
        ffn_out = self.ffn(self.norm2(x), training=training)
        x = x + self.drop2(ffn_out, training=training)
        return x


def build_tabular_transformer(
    num_features: int,
    num_classes: int,
    embed_dim: int = EMBED_DIM,
    num_heads: int = NUM_HEADS,
    ff_dim: int = FF_DIM,
    num_layers: int = NUM_LAYERS,
    dropout_rate: float = DROPOUT_RATE,
) -> Model:
    """Build a tabular transformer where each feature is treated as a token."""
    inputs = layers.Input(shape=(num_features, 1), name="feature_tokens")

    # Project each token from dim 1 to embed_dim
    x = layers.Dense(embed_dim, name="token_projection")(inputs)

    # Stack num_layers transformer blocks
    for i in range(num_layers):
        x = TransformerBlock(
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate,
            name=f"transformer_block_{i}",
        )(x)

    # Aggregate across the feature/token dimension
    x = layers.GlobalAveragePooling1D(name="global_avg_pool")(x)
    x = layers.Dropout(dropout_rate, name="head_dropout")(x)

    # Classification head
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(x)

    return Model(inputs=inputs, outputs=outputs, name="TabularTransformer")


model = build_tabular_transformer(
    num_features=num_features,
    num_classes=num_classes,
)
model.summary()

## 10. Compile

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="acc")],
)

## 11. Build tf.data Pipelines

In [ ]:
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=SHUFFLE_BUFFER_SIZE, seed=SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Free raw arrays now that tf.data holds the data
del X_train, y_train, X_val, y_val
gc.collect()

## 12. Callbacks

In [ ]:
os.makedirs(ARTIFACT_DIR, exist_ok=True)
checkpoint_path = os.path.join(ARTIFACT_DIR, "best_model.keras")

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE_ES,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=PATIENCE_LR,
        min_lr=1e-6,
        verbose=1,
    ),
]
print(f"Artifacts will be saved to: {os.path.abspath(ARTIFACT_DIR)}")

## 13. Training

In [ ]:
history = model.fit(
    train_ds,
    epochs=MAX_EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

In [ ]:
hist_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_df["loss"], label="train")
axes[0].plot(hist_df["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(hist_df["acc"], label="train")
axes[1].plot(hist_df["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("Training History — CSE-CIC-IDS2018 Improved Transformer")
plt.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, "training_history.png"), dpi=150)
plt.show()

## 14. Evaluation

In [ ]:
test_ds = (
    tf.data.Dataset.from_tensor_slices(X_test)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

y_pred_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=-1)

del y_pred_prob, test_ds, X_test
gc.collect()

print(f"y_pred shape : {y_pred.shape}")
print(f"y_test shape : {y_test.shape}")

In [ ]:
# ── Per-class report ──────────────────────────────────────────────────────────
print("=" * 70)
print("Classification Report")
print("=" * 70)
print(classification_report(y_test, y_pred, digits=4, target_names=class_names))

# ── Aggregate metrics ─────────────────────────────────────────────────────────
macro_f1      = f1_score(y_test, y_pred, average="macro", zero_division=0)
balanced_acc  = balanced_accuracy_score(y_test, y_pred)

print(f"Macro F1-Score      : {macro_f1:.4f}")
print(f"Balanced Accuracy   : {balanced_acc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=ax,
)
ax.set_title("Confusion Matrix — CSE-CIC-IDS2018 Improved Transformer")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
fig.savefig(os.path.join(ARTIFACT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 15. Save Artifacts

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
model_path = os.path.join(ARTIFACT_DIR, "model_final.keras")
model.save(model_path)
print(f"Model saved          : {model_path}")

# ── Scaler ────────────────────────────────────────────────────────────────────
scaler_path = os.path.join(ARTIFACT_DIR, "scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"Scaler saved         : {scaler_path}")

# ── Label classes ─────────────────────────────────────────────────────────────
label_path = os.path.join(ARTIFACT_DIR, "label_classes.json")
with open(label_path, "w") as fh:
    json.dump({"classes": class_names}, fh, indent=2)
print(f"Label classes saved  : {label_path}")

# ── Training history ──────────────────────────────────────────────────────────
history_path = os.path.join(ARTIFACT_DIR, "training_history.json")
with open(history_path, "w") as fh:
    json.dump(history.history, fh, indent=2)
print(f"Training history saved: {history_path}")

# ── Summary metrics ───────────────────────────────────────────────────────────
metrics_path = os.path.join(ARTIFACT_DIR, "eval_metrics.json")
with open(metrics_path, "w") as fh:
    json.dump(
        {
            "macro_f1": round(float(macro_f1), 6),
            "balanced_accuracy": round(float(balanced_acc), 6),
        },
        fh,
        indent=2,
    )
print(f"Eval metrics saved   : {metrics_path}")

print("\nAll artifacts written to:", os.path.abspath(ARTIFACT_DIR))